# 迷路メーカー＆脱出ロボット
全体の完成例。自分の作品との比較用。
説明サイト: https://hirasunaryou.github.io/for_KT_family/study/programming/python-maze-robot/
maze_tools.pyを同じフォルダへ。

## MISSION 1: 文字の地図が、迷路になる

In [ ]:
grid = (
    "#############",
    "#.....#.....#",
    "#.###.#.###.#",
    "#...#...#...#",
    "###.#####.#.#",
    "#...#.....#.#",
    "#.###.#####.#",
    "#.....#.....#",
    "#.#####.###.#",
    "#...........#",
    "#############",
)
start = (1, 1)
goal = (11, 1)
robot = (1, 9)

import matplotlib.pyplot as plt

pixels = []
for row in grid:
    values = []
    for tile in row:
        if tile == "#":
            values.append(0)
        else:
            values.append(1)
    pixels.append(values)

fig, ax = plt.subplots()
ax.imshow(pixels, cmap="gray", vmin=0, vmax=1, origin="upper")
ax.set_aspect("equal")
ax.set_title("My first maze")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

In [ ]:
from maze_tools import draw_maze

# 目印と座標を整える完成部品。ここはコピー可
draw_maze(grid, player=start, goal=goal, robot=robot)

予想:

確かめたこと:

改造したこと:

## MISSION 2: 壁をすり抜けないプレイヤー

In [ ]:
def can_walk(grid, pos):
    x, y = pos
    height = len(grid)
    width = len(grid[0])
    return 0 <= x < width and 0 <= y < height and grid[y][x] == "."

def neighbors(grid, pos):
    x, y = pos
    result = []
    for dx, dy in [(1, 0), (0, 1), (-1, 0), (0, -1)]:
        candidate = (x + dx, y + dy)
        if can_walk(grid, candidate):
            result.append(candidate)
    return result

def move_player(grid, pos, direction):
    moves = {"right": (1, 0), "down": (0, 1), "left": (-1, 0), "up": (0, -1)}
    dx, dy = moves[direction]
    candidate = (pos[0] + dx, pos[1] + dy)
    if can_walk(grid, candidate):
        return candidate
    return pos

In [ ]:
from maze_tools import show_walker

panel = show_walker(grid, start, goal, move_player)

予想:

確かめたこと:

改造したこと:

## MISSION 3: 足跡を覚えるロボット

In [ ]:
import random

def wander(grid, start, goal, seed=0, max_steps=200):
    rng = random.Random(seed)
    pos = start
    seen = {start}
    history = [start]
    for step in range(max_steps):
        if pos == goal:
            break
        choices = neighbors(grid, pos)
        if not choices:
            break
        fresh = []
        for candidate in choices:
            if candidate not in seen:
                fresh.append(candidate)
        pos = rng.choice(fresh if fresh else choices)
        seen.add(pos)
        history.append(pos)
    return history, seen

In [ ]:
from maze_tools import animate_search

history, seen = wander(grid, start, goal, seed=3)
viewer = animate_search(grid, history, [], goal=goal, label="WALK")

予想:

確かめたこと:

改造したこと:

## MISSION 4: 最短経路が光る探索

In [ ]:
from collections import deque

def find_path(grid, start, goal):
    if not can_walk(grid, start) or not can_walk(grid, goal):
        return [], []
    queue = deque([start])
    parents = {start: None}
    order = []
    while queue:
        pos = queue.popleft()
        order.append(pos)
        if pos == goal:
            path = []
            while pos is not None:
                path.append(pos)
                pos = parents[pos]
            path.reverse()
            return path, order
        for candidate in neighbors(grid, pos):
            if candidate not in parents:
                parents[candidate] = pos
                queue.append(candidate)
    return [], order

In [ ]:
from maze_tools import animate_search

path, order = find_path(grid, start, goal)
print("調べたマス:", len(order))
print("最短歩数:", len(path) - 1 if path else "道なし")
viewer = animate_search(grid, order, path, goal=goal)

予想:

確かめたこと:

改造したこと:

## MISSION 5: 遊べる迷路だけを自動生成

In [ ]:
import random

def make_maze(width=13, height=11, wall_rate=0.30, seed=0):
    if type(width) is not int or type(height) is not int or not 7 <= width <= 41 or not 7 <= height <= 41:
        raise ValueError("幅と高さは7〜41の整数にしよう")
    if not 0 <= wall_rate <= 1:
        raise ValueError("壁の割合は0〜1にしよう")
    rng = random.Random(seed)
    start = (1, 1)
    goal = (width - 2, 1)
    robot = (1, height - 2)
    for attempt in range(200):
        rows = []
        for y in range(height):
            row = []
            for x in range(width):
                if x == 0 or y == 0 or x == width - 1 or y == height - 1:
                    row.append("#")
                else:
                    row.append("#" if rng.random() < wall_rate else ".")
            rows.append(row)
        for x, y in [start, goal, robot]:
            rows[y][x] = "."
        candidate = tuple("".join(row) for row in rows)
        path, _ = find_path(candidate, start, goal)
        robot_path, _ = find_path(candidate, robot, start)
        if path and robot_path:
            return candidate
    raise RuntimeError("200回作ってもつながらない。壁の割合を下げてみよう")

予想:

確かめたこと:

改造したこと:

## MISSION 6: 追跡ロボットから脱出せよ

In [ ]:
def new_game(grid, start, goal, robot, robot_every=2):
    return {"grid": tuple(grid), "player": start, "goal": goal, "robot": robot,
            "steps": 0, "robot_every": robot_every, "status": "playing",
            "message": "ゴールへ向かおう！"}

def take_turn(state, direction):
    s = dict(state)
    if s["status"] != "playing":
        return s
    if direction == "wait":
        pos = s["player"]
    else:
        pos = move_player(s["grid"], s["player"], direction)
        if pos == s["player"]:
            s["message"] = "壁には進めない。手数は増えないよ"
            return s
    s["player"] = pos
    s["steps"] += 1
    if pos == s["robot"]:
        s["status"] = "lost"
        s["message"] = "ロボットにぶつかった！"
        return s
    if pos == s["goal"]:
        s["status"] = "won"
        s["message"] = "脱出成功！"
        return s
    if s["steps"] % s["robot_every"] == 0:
        path, _ = find_path(s["grid"], s["robot"], pos)
        if len(path) >= 2:
            s["robot"] = path[1]
    if s["robot"] == pos:
        s["status"] = "lost"
        s["message"] = "つかまった！ 次はルートを変えよう"
    else:
        s["message"] = "次はどう動く？"
    return s

In [ ]:
from maze_tools import show_game

# 関数セルを先に実行。接続部分はコピー可
initial = new_game(grid, start, goal, robot, robot_every=2)
panel = show_game(initial, take_turn)

# 自動生成した地図を使うなら別のセルで
# board = load_maze("my_maze.json")
# initial = new_game(board, (1, 1), (len(board[0])-2, 1), (1, len(board)-2))
# panel = show_game(initial, take_turn)

予想:

確かめたこと:

改造したこと: